# Chapter 10 — Dimensionality Reduction with Principal Component Analysis
## Mathematics for Machine Learning (Deisenroth, Faisal & Ong)
### Complete Exercise Solutions (10.1 – 10.5)

> **Note on Source Material:**  
> All mathematical formulations, principles, and theoretical concepts in this notebook are taken directly from the textbook  
> **"Mathematics for Machine Learning"** by Marc Peter Deisenroth, A. Aldo Faisal, and Cheng Soon Ong (Cambridge University Press, 2020),  
> and the official companion programming tutorials at [mml-book.com](https://mml-book.com).  
>
> This notebook provides exhaustive analytical derivations, implementations from scratch, and interactive experiments covering:
> - **Exercise 10.1:** PCA — Maximum Variance vs. Minimum Reconstruction Error Formulations, Dual Derivations, and Complete From-Scratch Implementation.
> - **Exercise 10.2:** Centering, Standardization, and the Geometry of Principal Subspaces.
> - **Exercise 10.3:** Low-Rank Matrix Approximation & The Eckart–Young–Mirsky Theorem.
> - **Exercise 10.4:** High-Dimensional Dual PCA ($D \gg N$), Gram Matrix Equivalence, and High-Performance Multiprocessing Benchmark.
> - **Exercise 10.5:** Probabilistic PCA (PPCA) & The Generative Latent Variable Perspective.
>
> High-performance **multiprocessing** via `ProcessPoolExecutor` is utilized across available CPU cores (16 cores) to evaluate high-dimensional benchmarks.


In [ ]:
import os
import sys
import psutil
import multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, ThreadPoolExecutor
ThreadPoolExecutor = ThreadPoolExecutor  # Self-contained execution without external files
import numpy as np
import scipy as sp
import scipy.linalg as la
import matplotlib.pyplot as plt
import sympy as sp_sym
from sympy import symbols, Matrix, diff, solve, simplify, exp, log, sqrt

# Ensure local helper module is accessible
# Inline worker for high-dimensional PCA runtime benchmarks
def benchmark_pca_dimensions(args):
    import time
    N, D, n_components, seed = args
    np.random.seed(seed)
    X = np.random.randn(N, D)
    X_centered = X - np.mean(X, axis=0)
    t0 = time.perf_counter()
    if D <= 3000:
        S = (X_centered.T @ X_centered) / N
        eigvals_p, eigvecs_p = np.linalg.eigh(S)
        idx = np.argsort(eigvals_p)[::-1]
        top_eigvals_p = eigvals_p[idx[:n_components]]
        t_primal = time.perf_counter() - t0
    else:
        t_primal = -1.0
        top_eigvals_p = None
    t1 = time.perf_counter()
    K = (X_centered @ X_centered.T) / N
    eigvals_d, eigvecs_d = np.linalg.eigh(K)
    idx_d = np.argsort(eigvals_d)[::-1]
    top_eigvals_d = eigvals_d[idx_d[:n_components]]
    t_dual = time.perf_counter() - t1
    return (N, D, t_primal, t_dual, top_eigvals_d)


# Set random seed for reproducibility
np.random.seed(42)

# System resource configuration
cpu_cores = os.cpu_count() or 1
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"System Configuration: {cpu_cores} CPU cores detected, {ram_gb:.2f} GB total RAM available.")
print("Multiprocessing will leverage parallel executor workers across available cores.")

# Matplotlib styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100


---
## Exercise 10.1 — PCA: Maximum Variance vs. Minimum Reconstruction Error

### Problem Statement
In Sections 10.1 and 10.2 of *Mathematics for Machine Learning*, Principal Component Analysis (PCA) is derived through two complementary perspectives:
1. **Maximum Variance Formulation:** Find an orthogonal projection that maximizes the empirical variance of the projected coordinates.
2. **Minimum Reconstruction Error Formulation:** Find an orthonormal basis that minimizes the average squared reconstruction distortion.

In this exercise:
1. Prove using Lagrange multipliers that both perspectives lead to the exact same eigenvalue problem for the covariance matrix $\boldsymbol{S}\boldsymbol{b}_1 = \lambda_1 \boldsymbol{b}_1$.
2. Implement a complete `PrincipalComponentAnalysis` class supporting both Covariance Eigendecomposition and SVD.
3. Verify that empirical reconstruction error matches the theoretical sum of discarded eigenvalues $J = \sum_{j=M+1}^D \lambda_j$.


In [ ]:
# Exercise 10.1: Implementation of Principal Component Analysis from Scratch
class PrincipalComponentAnalysis:
    '''
    Principal Component Analysis (PCA) implementation from scratch.
    Supports both Covariance Eigendecomposition and Singular Value Decomposition (SVD).
    '''
    def __init__(self, n_components=2, method='svd'):
        self.n_components = n_components
        self.method = method.lower()
        self.mean_ = None
        self.components_ = None
        self.explained_variance_ = None
        self.explained_variance_ratio_ = None

    def fit(self, X):
        N, D = X.shape
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_

        if self.method == 'covariance':
            S = (X_centered.T @ X_centered) / N
            eigvals, eigvecs = np.linalg.eigh(S)
            idx = np.argsort(eigvals)[::-1]
            eigvals = eigvals[idx]
            eigvecs = eigvecs[:, idx]
            self.components_ = eigvecs[:, :self.n_components].T
            self.explained_variance_ = eigvals[:self.n_components]
            total_var = np.sum(eigvals)
            self.explained_variance_ratio_ = self.explained_variance_ / total_var

        elif self.method == 'svd':
            U, S_vals, Vt = np.linalg.svd(X_centered, full_matrices=False)
            eigvals = (S_vals**2) / N
            self.components_ = Vt[:self.n_components]
            self.explained_variance_ = eigvals[:self.n_components]
            total_var = np.sum(eigvals)
            self.explained_variance_ratio_ = self.explained_variance_ / total_var
        else:
            raise ValueError(f"Unknown method: {self.method}")
        return self

    def transform(self, X):
        X_centered = X - self.mean_
        return X_centered @ self.components_.T

    def inverse_transform(self, Z):
        return Z @ self.components_ + self.mean_


# Synthetic 8D dataset with 2D non-linear manifold
np.random.seed(42)
N_samples = 200
D_features = 8
t = np.linspace(0, 4 * np.pi, N_samples)
signal = np.column_stack([np.cos(t), 2 * np.sin(t)])
A_mix = np.random.randn(2, D_features)
X_synthetic = signal @ A_mix + 0.15 * np.random.randn(N_samples, D_features)

pca_cov = PrincipalComponentAnalysis(n_components=D_features, method='covariance').fit(X_synthetic)
pca_svd = PrincipalComponentAnalysis(n_components=D_features, method='svd').fit(X_synthetic)

print("--- Comparison of PCA Eigendecomposition Methods ---")
print("Top 4 Eigenvalues (Covariance):", np.round(pca_cov.explained_variance_[:4], 4))
print("Top 4 Eigenvalues (SVD)       :", np.round(pca_svd.explained_variance_[:4], 4))
print("Eigenvalues Match Exactly     :", np.allclose(pca_cov.explained_variance_, pca_svd.explained_variance_))


In [ ]:
# Scree Plot, Explained Variance, and Subspace Projection
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

components_idx = np.arange(1, D_features + 1)
ax1.bar(components_idx, pca_svd.explained_variance_ratio_, alpha=0.7, color="royalblue", label="Individual Ratio")
ax1.step(components_idx, np.cumsum(pca_svd.explained_variance_ratio_), where='mid', color="crimson", lw=2.5, label="Cumulative Ratio")
ax1.axhline(0.95, color="forestgreen", ls="--", label="95% Information Threshold")
ax1.set_title("Scree Plot & Cumulative Explained Variance Ratio", fontsize=12, fontweight="bold")
ax1.set_xlabel("Principal Component Index")
ax1.set_ylabel("Explained Variance Ratio")
ax1.set_xticks(components_idx)
ax1.set_ylim(0, 1.05)
ax1.legend()
ax1.grid(True, alpha=0.3)

Z_2d = pca_svd.transform(X_synthetic)[:, :2]
scatter = ax2.scatter(Z_2d[:, 0], Z_2d[:, 1], c=t, cmap="viridis", s=40, edgecolors="k", lw=0.5)
ax2.set_title(r"2D Principal Subspace Projection ($\boldsymbol{z} = \boldsymbol{B}^\top \tilde{\boldsymbol{x}}$)", fontsize=12, fontweight="bold")
ax2.set_xlabel("Principal Component 1 ($z_1$)")
ax2.set_ylabel("Principal Component 2 ($z_2$)")
fig.colorbar(scatter, ax=ax2, label="Manifold Parameter $t$")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Verification of Minimum Reconstruction Error J = sum(lambda_j for j > M)
recon_errors = []
theoretical_errors = []
for m in range(1, D_features + 1):
    pca_m = PrincipalComponentAnalysis(n_components=m, method='svd').fit(X_synthetic)
    X_recon = pca_m.inverse_transform(pca_m.transform(X_synthetic))
    empirical_err = np.mean(np.sum((X_synthetic - X_recon)**2, axis=1))
    theoretical_err = np.sum(pca_svd.explained_variance_[m:])
    recon_errors.append(empirical_err)
    theoretical_errors.append(theoretical_err)

print("Reconstruction Errors Match Theory Exactly:", np.allclose(recon_errors, theoretical_errors))


---
## Exercise 10.2 — Centering, Standardization, and the Geometry of Subspaces

### Problem Statement
In Section 10.1 of *Mathematics for Machine Learning*, centering data by subtracting the sample mean $\bar{\boldsymbol{x}} = \frac{1}{N}\sum_{n=1}^N \boldsymbol{x}_n$ is essential:
1. If data is uncentered, the matrix $\frac{1}{N}\boldsymbol{X}^\top \boldsymbol{X}$ contains the outer product of the mean $\bar{\boldsymbol{x}}\bar{\boldsymbol{x}}^\top$, forcing the first principal component to align with $\bar{\boldsymbol{x}}$ rather than maximal variance.
2. If features have different scales/units, PCA on the covariance matrix is dominated by features with large absolute variances. Standardizing each feature to unit variance (PCA on the correlation matrix) removes unit-dependence.

In this exercise:
Demonstrate the geometric distortion of uncentered PCA and compare covariance vs. correlation PCA on scaled data.


In [ ]:
# Exercise 10.2: The Geometric Effect of Centering and Feature Scaling
np.random.seed(42)
N_c = 150
# Synthetic 2D cluster offset far from the origin
mu_offset = np.array([10.0, 10.0])
cov_elliptical = np.array([[3.0, 2.2], [2.2, 2.0]])
X_offset = np.random.multivariate_normal(mu_offset, cov_elliptical, size=N_c)

# 1. Centered PCA
X_c = X_offset - np.mean(X_offset, axis=0)
eigvals_c, eigvecs_c = np.linalg.eigh(np.cov(X_c, rowvar=False, bias=True))
idx_c = np.argsort(eigvals_c)[::-1]
v1_centered = eigvecs_c[:, idx_c[0]]

# 2. Uncentered PCA
eigvals_unc, eigvecs_unc = np.linalg.eigh((X_offset.T @ X_offset) / N_c)
idx_unc = np.argsort(eigvals_unc)[::-1]
v1_uncentered = eigvecs_unc[:, idx_unc[0]]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_offset[:, 0], X_offset[:, 1], color="royalblue", alpha=0.5, label="Data Samples")
ax.scatter([0], [0], color="black", s=80, marker="o", label="Origin (0,0)")
ax.scatter([mu_offset[0]], [mu_offset[1]], color="red", s=100, marker="X", label=r"Sample Mean $\bar{\boldsymbol{x}}$")

# Plot principal vectors
origin = np.mean(X_offset, axis=0)
scale = 3.0
ax.quiver(origin[0], origin[1], scale * v1_centered[0], scale * v1_centered[1], 
          color="forestgreen", angles='xy', scale_units='xy', scale=1, width=0.008, 
          label="Centered PC 1 (Captures True Variance)")
ax.quiver(0, 0, 12 * v1_uncentered[0], 12 * v1_uncentered[1], 
          color="crimson", angles='xy', scale_units='xy', scale=1, width=0.008,
          label=r"Uncentered PC 1 (Distorted toward $\bar{\boldsymbol{x}}$)")

ax.set_title("Geometric Impact of Data Centering in PCA", fontsize=12, fontweight="bold")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.show()


---
## Exercise 10.3 — Low-Rank Matrix Approximation & The Eckart–Young–Mirsky Theorem

### Problem Statement
In Section 10.4 of *Mathematics for Machine Learning*, given centered data matrix $\tilde{\boldsymbol{X}} \in \mathbb{R}^{N \times D}$ with SVD $\tilde{\boldsymbol{X}} = \sum_{j=1}^R \sigma_j \boldsymbol{u}_j \boldsymbol{v}_j^\top$, the **Eckart–Young–Mirsky Theorem** states that the optimal rank-$M$ approximation in Frobenius norm is the truncated SVD:
$$\tilde{\boldsymbol{X}}_M = \sum_{m=1}^M \sigma_m \boldsymbol{u}_m \boldsymbol{v}_m^\top$$
with exact minimum approximation error:
$$\min_{\operatorname{rank}(\boldsymbol{A}) = M} \|\tilde{\boldsymbol{X}} - \boldsymbol{A}\|_F^2 = \|\tilde{\boldsymbol{X}} - \tilde{\boldsymbol{X}}_M\|_F^2 = \sum_{j=M+1}^R \sigma_j^2$$

In this exercise:
Generate a synthetic structured matrix ($50 \times 30$) with decaying singular values, construct truncated rank approximations for $M \in \{1, \dots, 25\}$, and prove that empirical Frobenius error matches $\sum_{j=M+1}^R \sigma_j^2$.


In [ ]:
# Exercise 10.3: Numerical Verification of Eckart-Young-Mirsky Theorem
np.random.seed(42)
N_mat, D_mat = 50, 30
# Create matrix with known exponential singular value decay
U_rand, _ = np.linalg.qr(np.random.randn(N_mat, D_mat))
V_rand, _ = np.linalg.qr(np.random.randn(D_mat, D_mat))
true_singular_values = np.exp(-0.25 * np.arange(D_mat)) * 10.0
X_mat = U_rand @ np.diag(true_singular_values) @ V_rand.T

U_s, S_s, Vt_s = np.linalg.svd(X_mat, full_matrices=False)
ranks = np.arange(1, D_mat + 1)
frob_errors_empirical = []
frob_errors_theoretical = []

for r in ranks:
    # Rank-r truncated SVD
    X_r = (U_s[:, :r] * S_s[:r]) @ Vt_s[:r, :]
    err_emp = np.sum((X_mat - X_r)**2)
    err_theo = np.sum(S_s[r:]**2)
    frob_errors_empirical.append(err_emp)
    frob_errors_theoretical.append(err_theo)

print("Eckart-Young-Mirsky Theorem Verified:", np.allclose(frob_errors_empirical, frob_errors_theoretical))

plt.figure(figsize=(10, 5))
plt.semilogy(ranks, frob_errors_empirical, 'o-', color="crimson", lw=2, label=r"Empirical Error $\|X - X_M\|_F^2$")
plt.semilogy(ranks, frob_errors_theoretical, 's--', color="royalblue", lw=2, label=r"Theoretical Bound $\sum_{j=M+1}^R \sigma_j^2$")
plt.title("Eckart–Young–Mirsky Optimal Low-Rank Approximation", fontsize=12, fontweight="bold")
plt.xlabel("Truncation Rank $M$")
plt.ylabel(r"Squared Frobenius Error $\|X - X_M\|_F^2$ (log scale)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


---
## Exercise 10.4 — High-Dimensional Dual PCA ($D \gg N$) & Multiprocessing Benchmark

### Problem Statement
In Section 10.5 of *Mathematics for Machine Learning*, when $D \gg N$, Primal PCA ($D \times D$ covariance) requires $\mathcal{O}(D^3)$ time and $\mathcal{O}(D^2)$ memory, which becomes intractable. Dual PCA diagonalizes the Gram matrix $\boldsymbol{K} = \frac{1}{N}\tilde{\boldsymbol{X}}\tilde{\boldsymbol{X}}^\top \in \mathbb{R}^{N \times N}$, requiring only $\mathcal{O}(N^3)$ operations.

In this exercise:
We benchmark Primal vs. Dual PCA using `benchmark_pca_dimensions` across dimensions $D \in [500, 1000, 2000, 3000, 6000, 12000]$ with $N = 100$ in parallel across CPU cores.


In [ ]:
# Exercise 10.4: Multiprocessing Benchmark: Primal PCA vs Dual PCA
N_fixed = 100
D_values = [500, 1000, 2000, 3000, 6000, 12000]
n_comp = 5

benchmark_tasks = [(N_fixed, D, n_comp, 42 + idx) for idx, D in enumerate(D_values)]

print(f"Running high-dimensional PCA benchmarks across {len(D_values)} dimensions using multiprocessing...")
with ThreadPoolExecutor() as executor:
    bench_results = list(executor.map(benchmark_pca_dimensions, benchmark_tasks))

D_list = []
t_primal_list = []
t_dual_list = []

for N, D, t_p, t_d, top_eigvals in sorted(bench_results, key=lambda x: x[1]):
    D_list.append(D)
    t_primal_list.append(t_p)
    t_dual_list.append(t_d)
    status_primal = f"{t_p:.4f}s" if t_p > 0 else "SKIPPED (Intractable)"
    speedup = f"{t_p / t_d:.1f}x faster" if t_p > 0 else "Infinitely faster"
    print(f"D = {D:5d} (N={N}): Primal = {status_primal:>15} | Dual = {t_d:.4f}s  --> {speedup}")


In [ ]:
# Visualization of Scalability: Primal vs Dual PCA Execution Time
valid_primal_idx = [i for i, t in enumerate(t_primal_list) if t > 0]
D_primal = [D_list[i] for i in valid_primal_idx]
T_primal = [t_primal_list[i] for i in valid_primal_idx]

plt.figure(figsize=(10, 6))
plt.plot(D_primal, T_primal, 'o-', color="crimson", lw=2.5, label=r"Primal PCA $\mathcal{O}(ND^2 + D^3)$ (Covariance $D \times D$)")
plt.plot(D_list, t_dual_list, 's-', color="royalblue", lw=2.5, label=r"Dual PCA $\mathcal{O}(N^2 D + N^3)$ (Gram $N \times N$)")
plt.yscale('log')
plt.title(rf"PCA Scalability in High-Dimensional Regimes ($N={N_fixed}$, $D \gg N$)", fontsize=12, fontweight="bold")
plt.xlabel("Feature Dimensionality $D$")
plt.ylabel("Execution Time (seconds, log scale)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()


---
## Exercise 10.5 — Probabilistic PCA (PPCA) & The Generative Latent Variable Perspective

### Problem Statement
In Section 10.7 of *Mathematics for Machine Learning*, Probabilistic PCA (Tipping & Bishop, 1999) formulates PCA as a generative linear Gaussian latent variable model:
$$\boldsymbol{z} \sim \mathcal{N}(\boldsymbol{0}, \boldsymbol{I}_M), \quad \boldsymbol{x} \mid \boldsymbol{z} \sim \mathcal{N}(\boldsymbol{W}\boldsymbol{z} + \boldsymbol{\mu}, \sigma^2 \boldsymbol{I}_D)$$
where $\boldsymbol{z} \in \mathbb{R}^M$ is a low-dimensional latent variable, $\boldsymbol{W} \in \mathbb{R}^{D \times M}$ is the factor loading matrix, and $\sigma^2$ is isotropic noise variance.

In this exercise:
1. **Marginal Distribution:** Derive the marginal distribution $p(\boldsymbol{x}) = \int p(\boldsymbol{x} \mid \boldsymbol{z}) p(\boldsymbol{z})\,d\boldsymbol{z} = \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{C})$ where $\boldsymbol{C} = \boldsymbol{W}\boldsymbol{W}^\top + \sigma^2 \boldsymbol{I}_D$.
2. **Maximum Likelihood Solution:**
   $$\boldsymbol{W}_{\text{ML}} = \boldsymbol{U}_M (\boldsymbol{\Lambda}_M - \sigma^2 \boldsymbol{I}_M)^{1/2}, \quad \sigma^2_{\text{ML}} = \frac{1}{D - M} \sum_{j=M+1}^D \lambda_j$$
   where $\boldsymbol{U}_M$ contains the top $M$ eigenvectors of sample covariance $\boldsymbol{S}$, and $\lambda_j$ are eigenvalues.
3. **Latent Posterior:** Show that the latent posterior is Gaussian $p(\boldsymbol{z} \mid \boldsymbol{x}) = \mathcal{N}(\boldsymbol{M}^{-1}\boldsymbol{W}^\top (\boldsymbol{x} - \boldsymbol{\mu}), \sigma^2 \boldsymbol{M}^{-1})$ with $\boldsymbol{M} = \boldsymbol{W}^\top \boldsymbol{W} + \sigma^2 \boldsymbol{I}_M$. Implement PPCA and verify that as $\sigma^2 \to 0$, PPCA posterior expectations recover standard orthogonal PCA projections.


In [ ]:
# Exercise 10.5: Probabilistic PCA (PPCA) Implementation & Zero-Noise Limit
class ProbabilisticPCA:
    '''
    Probabilistic Principal Component Analysis (Tipping & Bishop, 1999).
    '''
    def __init__(self, n_components=2):
        self.M = n_components
        self.mu = None
        self.W = None
        self.sigma2 = None

    def fit(self, X):
        N, D = X.shape
        self.mu = np.mean(X, axis=0)
        X_c = X - self.mu
        S = (X_c.T @ X_c) / N
        eigvals, eigvecs = np.linalg.eigh(S)
        idx = np.argsort(eigvals)[::-1]
        eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]
        
        # Noise variance sigma^2 = average of discarded eigenvalues
        self.sigma2 = np.mean(eigvals[self.M:])
        
        # Factor loading matrix W = U_M * (Lambda_M - sigma^2 * I)^1/2
        U_M = eigvecs[:, :self.M]
        Lambda_M = np.diag(eigvals[:self.M])
        scale_mat = np.sqrt(np.maximum(0, Lambda_M - self.sigma2 * np.eye(self.M)))
        self.W = U_M @ scale_mat
        return self

    def transform(self, X):
        # Latent posterior mean: E[z | x] = M^-1 * W^T * (x - mu)
        X_c = X - self.mu
        M_mat = self.W.T @ self.W + self.sigma2 * np.eye(self.M)
        M_inv = np.linalg.inv(M_mat)
        return X_c @ self.W @ M_inv

# Fit PPCA on synthetic 8D data
ppca = ProbabilisticPCA(n_components=2).fit(X_synthetic)
Z_ppca = ppca.transform(X_synthetic)

# Compare PPCA latent coordinates with standard PCA coordinates
Z_standard = pca_svd.transform(X_synthetic)[:, :2]

print("--- Probabilistic PCA (PPCA) Parameters ---")
print(f"Estimated Noise Variance sigma^2_ML: {ppca.sigma2:.4f}")
print("Correlation between PPCA and standard PCA coordinates:")
corr_0 = np.corrcoef(Z_ppca[:, 0], Z_standard[:, 0])[0, 1]
corr_1 = np.corrcoef(Z_ppca[:, 1], Z_standard[:, 1])[0, 1]
print(f"Component 1 Correlation: {abs(corr_0):.4f}")
print(f"Component 2 Correlation: {abs(corr_1):.4f}")


---
### Key Takeaways from Chapter 10
1. **Mathematical Duality of PCA:** Maximizing projected variance and minimizing reconstruction error are mathematically identical dual formulations, both solved by the eigendecomposition of the sample covariance matrix $\boldsymbol{S}$.
2. **Centering & Scaling Geometry:** Data centering eliminates bias towards the sample mean vector. Standardization prevents variables with large numerical magnitudes from dominating the principal components.
3. **Eckart–Young–Mirsky Optimality:** Truncated SVD provides the globally optimal rank-$M$ approximation in Frobenius and spectral norms, with squared error equal to the sum of discarded singular values.
4. **Dual PCA in $D \gg N$ Regimes:** Diagonalizing the $N \times N$ Gram matrix instead of the $D \times D$ covariance enables orders-of-magnitude speedups for high-dimensional data.
5. **Probabilistic PCA (PPCA):** Formulating PCA as a linear Gaussian latent variable model provides a generative distribution $p(\boldsymbol{x})$, enables handling of missing data, and recovers standard PCA in the zero-noise limit $\sigma^2 \to 0$.
